# 🎓 Supervised Fine-Tuning (SFT) with QLoRA for Educational RAG
### Model: `meta-llama/Meta-Llama-3-8B-Instruct`
### Frameworks: `transformers`, `peft`, `trl`, `bitsandbytes`, `accelerate`

---
### 📌 Overview
This notebook implements **4-bit Parameter-Efficient Fine-Tuning (QLoRA)** using Hugging Face's `trl.SFTTrainer`.
- **Target**: Fine-tunes `Meta-Llama-3-8B-Instruct` on educational Q&A data.
- **Hardware Requirement**: Runs on a **free Google Colab T4 GPU (15 GB VRAM)** or higher (A100, V100, L4).
- **Features**:
  - 4-bit NormalFloat (NF4) quantization with double quantization.
  - LoRA adapters on all attention and MLP projection layers (`q, k, v, o, gate, up, down`).
  - Llama-3 conversational prompt formatting (`apply_chat_template`).
  - Exports the trained LoRA adapter ready for Ollama or GGUF conversion.

In [ ]:
# Step 1: Install Required Libraries in Colab
!pip install -q --upgrade pip
!pip install -q transformers>=4.40.0 trl>=0.8.6 peft>=0.10.0 bitsandbytes>=0.43.0 accelerate>=0.29.0 datasets>=2.18.0

In [ ]:
# Step 2: Verify GPU Acceleration
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise RuntimeError("Please enable GPU acceleration in Colab: Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Step 3: Hugging Face Authentication
# Meta-Llama-3-8B-Instruct is a gated model. You must accept the license on Hugging Face
# and log in with your read-access Hugging Face token.
from huggingface_hub import login
from google.colab import userdata

# Option A: Colab Secrets (recommended)
try:
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face via Colab Secret 'HF_TOKEN'!")
except Exception:
    # Option B: Interactive Login
    print("Please enter your Hugging Face Token (with read permission):")
    login()

In [ ]:
# Step 4: Create Sample Educational SFT Dataset (JSONL)
import json

sft_data = [
    {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert educational AI tutor. Answer questions factually and strictly based on the provided context with citations."
            },
            {
                "role": "user",
                "content": "Context Chunks:\n[1] Isaac Newton published the Philosophiæ Naturalis Principia Mathematica in 1687, formulating the laws of motion and universal gravitation.\n\nQuestion: When and in what book did Newton introduce his laws of motion?"
            },
            {
                "role": "assistant",
                "content": "Sir Isaac Newton introduced his laws of motion in his foundational work, *Philosophiæ Naturalis Principia Mathematica*, published in 1687 [1]."
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert educational AI tutor. Answer questions factually and strictly based on the provided context with citations."
            },
            {
                "role": "user",
                "content": "Context Chunks:\n[1] Newton's First Law (Law of Inertia) states that an object remains at rest or in uniform motion unless acted upon by a net external force.\n[2] Newton's Second Law defines force as F = ma.\n\nQuestion: What does Newton's First Law state?"
            },
            {
                "role": "assistant",
                "content": "Newton's First Law, also known as the Law of Inertia, states that an object will continue in its state of rest or uniform motion in a straight line unless it is compelled to change that state by an external net force [1]."
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert educational AI tutor. Answer questions factually and strictly based on the provided context with citations."
            },
            {
                "role": "user",
                "content": "Context Chunks:\n[1] Newton's Third Law states that whenever one body exerts a force on a second body, the second body exerts an equal and opposite force on the first body.\n\nQuestion: Explain action-reaction forces in Newton's Third Law."
            },
            {
                "role": "assistant",
                "content": "According to Newton's Third Law, forces always occur in matched pairs: whenever an object exerts a force (action) on another, the second object simultaneously exerts an equal magnitude force in the opposite direction (reaction) on the first [1]."
            }
        ]
    },
    {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert educational AI tutor. Answer questions factually and strictly based on the provided context with citations."
            },
            {
                "role": "user",
                "content": "Context Chunks:\n[1] Photosynthesis is the process by which green plants transform light energy into chemical energy stored in glucose.\n\nQuestion: Who won the 1982 football World Cup?"
            },
            {
                "role": "assistant",
                "content": "I am sorry, but the provided context does not contain sufficient educational information to answer this question."
            }
        ]
    }
]

# Write to sft_dataset.jsonl
with open("sft_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in sft_data:
        f.write(json.dumps(item) + "\n")

print(f"Created sft_dataset.jsonl with {len(sft_data)} educational training examples.")

In [ ]:
# Step 5: Configure 4-bit QLoRA Quantization
import torch
from transformers import BitsAndBytesConfig

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # 4-bit NormalFloat for optimal weight distribution
    bnb_4bit_compute_dtype=compute_dtype, # bfloat16 or float16 for fast matrix ops
    bnb_4bit_use_double_quant=True,       # Nested quantization saving ~0.4 bits/param
)
print("BitsAndBytes 4-bit config successfully initialized.")

In [ ]:
# Step 6: Load Tokenizer and Base Model
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import prepare_model_for_kbit_training

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

print(f"Loading Tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Loading 4-bit Quantized Base Model {model_id}...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)

# Prepare model for LoRA training (freezes base model, enables gradient checkpointing)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False  # Disabled during training for gradient checkpointing
print("Base model loaded into VRAM successfully!")

In [ ]:
# Step 7: Configure LoRA Adapter (PEFT)
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,                              # LoRA rank dimension
    lora_alpha=32,                     # Scaling factor (typically 2 * r)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # Attention projections
        "gate_proj", "up_proj", "down_proj"       # MLP feed-forward projections
    ],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 8: Load and Format the Dataset with Llama-3 Chat Template
from datasets import load_dataset

dataset = load_dataset("json", data_files="sft_dataset.jsonl", split="train")

def format_conversations(batch):
    formatted_texts = []
    for messages in batch["messages"]:
        # Applies official <|begin_of_text|><|start_header_id|>... Llama 3 prompt structure
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}

dataset = dataset.map(format_conversations, batched=True)
print("Formatted sample text preview:\n")
print(dataset[0]["text"][:400] + "...\n")

In [ ]:
# Step 9: Configure and Launch SFTTrainer
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./sft_output",
    dataset_text_field="text",
    max_seq_length=512,
    per_device_train_batch_size=1,       # 1 or 2 fits easily on 15GB T4 GPU
    gradient_accumulation_steps=4,       # Effective batch size = 4
    optim="paged_adamw_8bit",            # Memory-efficient 8-bit optimizer
    learning_rate=2e-4,                  # Standard QLoRA learning rate
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=1,
    num_train_epochs=3,                  # Adjust based on real dataset size
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting Supervised Fine-Tuning...")
trainer.train()
print("SFT Training Completed Successfully!")

In [ ]:
# Step 10: Save Fine-Tuned LoRA Adapter
adapter_save_dir = "./educational_sft_adapter"

trainer.model.save_pretrained(adapter_save_dir)
tokenizer.save_pretrained(adapter_save_dir)

print(f"LoRA Adapter saved to '{adapter_save_dir}'!")
!ls -lh {adapter_save_dir}

In [ ]:
# Step 11: Inference Test on Fine-Tuned Model
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    temperature=0.2,
    do_sample=True,
    top_p=0.9,
)

test_messages = [
    {
        "role": "system",
        "content": "You are an expert educational AI tutor. Answer questions factually and strictly based on the provided context with citations."
    },
    {
        "role": "user",
        "content": "Context Chunks:\n[1] Isaac Newton published the Principia in 1687.\n\nQuestion: When was Newton's Principia published?"
    }
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt)
generated_text = outputs[0]["generated_text"]
print("Model Response:\n")
print(generated_text[len(prompt):])